## Installations | setup

In [8]:
pip install openai-agents

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import json
with open("..\\..\\playground\\japanese_email_test.json", encoding="utf-8") as f:
    emails = json.load(f)
print(f"Loaded {len(emails)} emails.")
for k, v in emails.items():
    print(f"  {k}  {v[:10].strip()}...")

Loaded 1 emails.
  email_01  差出人: 山本健太...


## Agent Creation

In [ ]:
from openai import AsyncAzureOpenAI
from agents import Agent, Runner, ModelSettings
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel

AZURE_API_KEY     = ""
AZURE_ENDPOINT    = ""
AZURE_API_VERSION = ""
AZURE_DEPLOYMENT  = ""

azure_client = AsyncAzureOpenAI(
    api_key=AZURE_API_KEY,
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
)

agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant",
    model=OpenAIChatCompletionsModel(
        model=AZURE_DEPLOYMENT,
        openai_client=azure_client,
    ),
    model_settings=ModelSettings(
        temperature=0.9,
        frequency_penalty=0.7,  # penalises tokens already used — reduces word-level repetition
        presence_penalty=0.6,   # penalises topics already mentioned — encourages new ideas
    ),
)

result = await Runner.run(agent, "Tell me a joke.")
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


Sure! Here's a joke for you:

Why don’t skeletons fight each other?  
Because they don’t have the guts! 😄


OPENAI_API_KEY is not set, skipping trace export


## Agent - Agent

In [11]:
email_reader_agent = Agent(
    name="Email Reader",
    instructions="You are an experienced email assistant that reads emails understands the intention, "
    "understands the ask and judges if all details are mentioned or not and mentioned the missed details in a seperate key array"
    "you will create the json as the output which will be having json with keys but no values, the values will be filled in next step"
    "You will cross-check number of available details in the email and respective keys available in the json internally not for output"
    "at the end you will be checking all the values are empty for string and empty json array or json object for nested keys if any"
    "except the json, don't give any other output in any case",

    model=OpenAIChatCompletionsModel(
        model=AZURE_DEPLOYMENT,
        openai_client=azure_client,
    ),
    model_settings=ModelSettings(temperature=0)
    )
# print(emails)
email_extraction_result = await Runner.run(email_reader_agent, emails["email_01"])
print(email_extraction_result.final_output)

OPENAI_API_KEY is not set, skipping trace export


```json
{
  "sender": "",
  "recipient": "",
  "date": "",
  "subject": "",
  "company_name": "",
  "contact_person": {
    "name": "",
    "position": "",
    "email": "",
    "phone": ""
  },
  "product_details": {
    "item_name": "",
    "model": "",
    "quantity": "",
    "budget": {
      "amount": "",
      "currency": "",
      "tax_included": "",
      "installation_included": "",
      "maintenance_contract": {
        "duration": "",
        "separate_quote": ""
      }
    },
    "delivery_schedule": [],
    "technical_specifications": {
      "dust_water_protection": "",
      "repeatability": "",
      "payload_capacity": "",
      "reach": "",
      "controller": ""
    },
    "compliance": {
      "laws": [],
      "standards": []
    }
  },
  "bidding_conditions": {
    "submission_deadline": "",
    "warranty_details": "",
    "inspection_schedule": {
      "factory_acceptance_test": "",
      "site_acceptance_test": ""
    },
    "spare_parts": {
      "list": [],
 

In [12]:
enhancer_agent = Agent(
    name="Enhancer Agent",
    instructions="You are an final reviewer and enhancer agent, " \
    "you will be getting the json with empty values and you will be filling the values in the json based on the details available in the email"
    "if you find any missing details you will be adding those details in the json and fill the values based on the details available in the email and also add the missing details in the missed_details key array"
    "make sure the final output json should have all the details filled and if any details are missing then it should be added in the missed_details key array with the details which are missing"
    "don't keep any irrelevent and unnecessary, coocked-up key-value pairs",

    model=OpenAIChatCompletionsModel(
        model=AZURE_DEPLOYMENT,
        openai_client=azure_client,
    ),
    model_settings=ModelSettings(temperature=0)
    )

enhancer_input = f"Email:\n{emails['email_01']}\n\nEmpty JSON Schema:\n{email_extraction_result.final_output}"
enhancer_result = await Runner.run(enhancer_agent, enhancer_input)
print(enhancer_result.final_output)

OPENAI_API_KEY is not set, skipping trace export


Here is the completed JSON schema based on the details provided in the email. Missing details have been added to the `missed_details` key array.

```json
{
  "sender": "山本健太 <yamamoto.kenta@suzuki-sekkei.co.jp>",
  "recipient": "bids@globalparts.com",
  "date": "2026-05-26T10:32:00",
  "subject": "【緊急】6軸アーク溶接ロボット調達依頼",
  "company_name": "鈴木設計株式会社",
  "contact_person": {
    "name": "山本健太",
    "position": "調達部長",
    "email": "yamamoto.kenta@suzuki-sekkei.co.jp",
    "phone": "052-432-8800（内線312）"
  },
  "product_details": {
    "item_name": "6軸アーク溶接ロボット",
    "model": "FANUCロボット M-10iA/12 または 安川電機 Motoman MA1440（同等品可）",
    "quantity": 18,
    "budget": {
      "amount": 220000000,
      "currency": "JPY",
      "tax_included": "No",
      "installation_included": "Yes",
      "maintenance_contract": {
        "duration": "3 years minimum",
        "separate_quote": "Yes"
      }
    },
    "delivery_schedule": [
      {
        "quantity": 6,
        "delivery_date": "2026-09-15"
   

In [13]:
import dspy

dspy.configure(
    lm=dspy.LM(
        model=f"azure/{AZURE_DEPLOYMENT}",
        api_key=AZURE_API_KEY,
        api_base=AZURE_ENDPOINT,
        api_version=AZURE_API_VERSION,
        temperature=0,
    )
)

class AuditInstructionGen(dspy.Signature):
    "Analyse the email and empty JSON schema, write field-by-field guidance for the auditor agent. Do NOT fill values."
    "add any fields that are missing but there in email, delete the fields which are not there in email but there in json schema, and also add the missing details in the missed_details key array with the details which are missing"
    "Make sure the relavent info about the product should be there in json, like if mobile brand is mentioned but there shuld be submodel if any, if not present move to additional fields required section in the json"
    "Ask the agent to follow the same language as in the email for the guidance, if the email is in japanese then the guidance should also be in japanese, if the email is in english then the guidance should also be in english and so on"
    email      = dspy.InputField(desc="Raw email text (may be in any language)")
    empty_json = dspy.InputField(desc="Empty JSON schema from the email reader agent")
    guidance   = dspy.OutputField(
        desc="Tailored, field-specific guidance telling the auditor agent how to fill each field, what to watch for, and how to handle missing values"
    )

class InstructionGenModule(dspy.Module):
    def __init__(self):
        self.gen = dspy.ChainOfThought(AuditInstructionGen)

    def forward(self, email, empty_json):
        return self.gen(email=email, empty_json=empty_json).guidance

instructor = InstructionGenModule()
dynamic_guidance = instructor(
    email=emails["email_01"],
    empty_json=email_extraction_result.final_output,
)

# DSPy-generated guidance + email + schema packaged as the agent user message
auditor_input = f"""Guidance:
{dynamic_guidance}

Email:
{emails["email_01"]}

Empty JSON Schema:
{email_extraction_result.final_output}"""

print("=== DSPy Generated Guidance ===")
print(dynamic_guidance)


OPENAI_API_KEY is not set, skipping trace export


=== DSPy Generated Guidance ===
### General Guidance:
- Ensure all fields are filled with the exact information provided in the email.
- If a field is not explicitly mentioned in the email, leave it blank or note it as "Not specified."
- Pay attention to units, currency, and formatting to ensure consistency.

### Field-Specific Guidance:
1. **sender**: Extract the sender's name and email address from the "差出人" line. Format as "山本健太 <yamamoto.kenta@suzuki-sekkei.co.jp>."

2. **recipient**: Use the email address in the "宛先" line: "bids@globalparts.com."

3. **date**: Use the date and time provided in the email header: "2026年5月26日（月曜日）午前10時32分." Convert to ISO 8601 format if required.

4. **subject**: Use the subject line: "【緊急】6軸アーク溶接ロボット調達依頼."

5. **company_name**: Extract from the signature block: "鈴木設計株式会社."

6. **contact_person**:
   - **name**: "山本健太"
   - **position**: "調達部長"
   - **email**: "yamamoto.kenta@suzuki-sekkei.co.jp"
   - **phone**: "052-432-8800（内線312）"

7. **product_de

In [14]:
auditor_writer_agent = Agent(
    name="Auditor Writer Agent",
    instructions="You are an auditor and writer agent. Follow the guidance provided to fill the JSON schema from the email. Return only clean, valid JSON - no markdown fences, no commentary.",
    model=OpenAIChatCompletionsModel(
        model=AZURE_DEPLOYMENT,
        openai_client=azure_client,
    ),
    model_settings=ModelSettings(temperature=0),
)

# --- WITHOUT DSPy: raw email + empty schema, no guidance ---
no_dspy_input = (
    "Email:\n" + emails["email_01"] + "\n\n" +
    "Empty JSON Schema:\n" + email_extraction_result.final_output
)
result_without_dspy = await Runner.run(auditor_writer_agent, no_dspy_input)

# --- WITH DSPy: DSPy-generated guidance prepended ---
result_with_dspy = await Runner.run(auditor_writer_agent, auditor_input)

print("=== WITHOUT DSPy ===")
print(result_without_dspy.final_output)
print("\n=== WITH DSPy ===")
print(result_with_dspy.final_output)


OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


=== WITHOUT DSPy ===
{
  "sender": "山本健太 <yamamoto.kenta@suzuki-sekkei.co.jp>",
  "recipient": "bids@globalparts.com",
  "date": "2026年5月26日（月曜日）午前10時32分",
  "subject": "【緊急】6軸アーク溶接ロボット調達依頼",
  "company_name": "鈴木設計株式会社",
  "contact_person": {
    "name": "山本健太",
    "position": "調達部長",
    "email": "yamamoto.kenta@suzuki-sekkei.co.jp",
    "phone": "052-432-8800（内線312）"
  },
  "product_details": {
    "item_name": "6軸アーク溶接ロボット",
    "model": "FANUCロボット M-10iA/12 または 安川電機 Motoman MA1440（同等品可）",
    "quantity": "18台",
    "budget": {
      "amount": "2億2千万円",
      "currency": "JPY",
      "tax_included": "No",
      "installation_included": "Yes",
      "maintenance_contract": {
        "duration": "3年",
        "separate_quote": "Yes"
      }
    },
    "delivery_schedule": [
      {
        "quantity": "6台",
        "date": "2026年9月15日"
      },
      {
        "quantity": "6台",
        "date": "2026年11月30日"
      },
      {
        "quantity": "6台",
        "date": "2027年1月31日"
    

In [15]:
import re

def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    try:
        return json.loads(match.group()) if match else {}
    except Exception:
        return {}

def count_fields(obj):
    filled, total = 0, 0
    if isinstance(obj, dict):
        for v in obj.values():
            f, t = count_fields(v)
            filled += f; total += t
    elif isinstance(obj, list):
        total += 1
        if obj:
            filled += 1
    else:
        total += 1
        if obj is not None and str(obj).strip() not in ("", "null"):
            filled += 1
    return filled, total

j_no  = extract_json(result_without_dspy.final_output)
j_yes = extract_json(result_with_dspy.final_output)

f_no,  t_no  = count_fields(j_no)
f_yes, t_yes = count_fields(j_yes)

pct_no  = f_no  / t_no  * 100 if t_no  else 0
pct_yes = f_yes / t_yes * 100 if t_yes else 0

print(f"{'Metric':<25} {'Without DSPy':>14}  {'With DSPy':>12}")
print("-" * 55)
print(f"{'Fields filled':<25} {f_no:>14}  {f_yes:>12}")
print(f"{'Total fields':<25} {t_no:>14}  {t_yes:>12}")
print(f"{'Coverage':<25} {pct_no:>13.1f}%  {pct_yes:>11.1f}%")
print()

# Show which fields differ
def flat_fields(obj, prefix=""):
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            out.update(flat_fields(v, f"{prefix}.{k}" if prefix else k))
    elif isinstance(obj, list):
        out[prefix] = bool(obj)
    else:
        out[prefix] = str(obj).strip() not in ("", "null", "None")
    return out

flat_no  = flat_fields(j_no)
flat_yes = flat_fields(j_yes)

gained = [k for k in flat_yes if flat_yes.get(k) and not flat_no.get(k)]
lost   = [k for k in flat_no  if flat_no.get(k)  and not flat_yes.get(k)]

if gained:
    print("Fields gained WITH DSPy:")
    for k in gained:
        print(f"  + {k}")
if lost:
    print("Fields lost WITH DSPy:")
    for k in lost:
        print(f"  - {k}")
if not gained and not lost:
    print("No field-level differences detected.")


Metric                      Without DSPy     With DSPy
-------------------------------------------------------
Fields filled                         37            37
Total fields                          37            37
Coverage                          100.0%        100.0%

No field-level differences detected.
